# 🧬 Auditor BI-RADS — RoBERTa Clínico + Datos Sintéticos (LLM)

**Experimento exploratorio (prueba de concepto).** Aumenta las clases de riesgo (BI-RADS 4, 5, 6) con informes sintéticos generados por un LLM, para evaluar si más datos de esas categorías mejoran al auditor.

**Reglas de validez (críticas):**
- Los datos sintéticos van **SOLO al entrenamiento**.
- El conjunto de **prueba es 100% real** → la métrica reportada es honesta.
- Los sintéticos **aún no están validados clínicamente**: este experimento decide si vale la pena conseguir esa validación (nivel 2).

**Gatillo de decisión:** si la mejora sobre el mejor modelo actual (RoBERTa clínico, F1-macro 0,5871) es sustancial → conseguir validación de un radiólogo. Si es marginal → la generación sintética simple no aporta en este corpus (hallazgo igualmente válido).

## 1 · Configuración y librerías

In [1]:
import torch, torch.nn as nn, numpy as np, pandas as pd, random, os, sys
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
sys.path.append('..')
from src.preprocessing import limpiar_texto, MAX_LENGTH

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
SYNTH_PATH='../data/synthetic/sinteticos.csv'
print(f"Dispositivo: {DEVICE}")

Dispositivo: mps


## 2 · Cargar datos reales y dividir (test 100% real)

El split se hace **solo sobre datos reales**, para que validación y prueba no contengan nada sintético.

In [2]:
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X = df['auditor_input'].values
y = df['BI-RADS'].values
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=SEED, stratify=y_tv)
print(f"Train real: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)} (todos reales)")

Train real: 3051 | Val: 652 | Test: 654 (todos reales)


## 3 · Cargar datos sintéticos e integrarlos SOLO a train

Se limpian con el mismo módulo (`limpiar_texto`) para que coincidan con la distribución real. Se añaden únicamente al conjunto de entrenamiento.

In [3]:
assert os.path.exists(SYNTH_PATH), f"No se encontró {SYNTH_PATH}. Genera el CSV con el LLM primero."
synth = pd.read_csv(SYNTH_PATH, encoding='utf-8')
assert {'texto','BI-RADS'}.issubset(synth.columns), "El CSV debe tener columnas: texto, BI-RADS"
synth = synth[synth['BI-RADS'].isin([4,5,6])].copy()
synth['texto_limpio'] = synth['texto'].apply(limpiar_texto)

print("Sintéticos cargados por clase:")
print(synth['BI-RADS'].value_counts().sort_index().to_string())

# Integrar SOLO a train
X_train_aug = np.concatenate([X_train, synth['texto_limpio'].values])
y_train_aug = np.concatenate([y_train, synth['BI-RADS'].values])
idx = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[idx], y_train_aug[idx]
print(f"\nTrain final: {len(X_train)} reales + {len(synth)} sintéticos = {len(X_train_aug)}")

Sintéticos cargados por clase:
BI-RADS
4    70
5    70
6    70

Train final: 3051 reales + 210 sintéticos = 3261


## 4 · Pesos de clase (sobre el train aumentado)

In [4]:
clases = np.unique(y_train_aug)
pesos = compute_class_weight(class_weight='balanced', classes=clases, y=y_train_aug)
peso_vec = np.ones(7, dtype=np.float32)
for c,w in zip(clases,pesos): peso_vec[c]=w
peso_tensor = torch.tensor(peso_vec, dtype=torch.float32).to(DEVICE)
print("Pesos por clase (0..6):", np.round(peso_vec,2))

Pesos por clase (0..6): [0.69 1.11 0.25 7.64 4.39 5.68 6.38]


## 5 · Tokenizador, Dataset y DataLoader

In [5]:
try: tokenizer = AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; tokenizer=AutoTokenizer.from_pretrained(MODELO)

class BIRADSDataset(Dataset):
    def __init__(self, t, l, tok, ml=MAX_LENGTH): self.t=list(t); self.l=list(l); self.tok=tok; self.ml=ml
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),truncation=True,padding='max_length',max_length=self.ml,
                   return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}

BATCH=16
train_loader=DataLoader(BIRADSDataset(X_train_aug,y_train_aug,tokenizer),batch_size=BATCH,shuffle=True)
val_loader  =DataLoader(BIRADSDataset(X_val,y_val,tokenizer),batch_size=BATCH)
test_loader =DataLoader(BIRADSDataset(X_test,y_test,tokenizer),batch_size=BATCH)
print(f"Batches -> train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

Batches -> train: 204 | val: 41 | test: 41


## 6 · Modelo (RoBERTa clínico) y configuración

In [6]:
class AuditorRoBERTa(nn.Module):
    def __init__(self, modelo, n_classes=7, dropout=0.5):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(modelo)
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(self.encoder.config.hidden_size, n_classes)
    def forward(self, input_ids, attention_mask):
        out=self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(out.last_hidden_state[:,0,:]))

model=AuditorRoBERTa(MODELO).to(DEVICE)
LR=3e-5; EPOCHS=15; PATIENCE=4
criterio=nn.CrossEntropyLoss(weight=peso_tensor)
optimizer=torch.optim.AdamW(model.parameters(), lr=LR)
scheduler=get_linear_schedule_with_warmup(optimizer,0,len(train_loader)*EPOCHS)
print("Configuración: RoBERTa clínico + CrossEntropy ponderada (misma config ganadora)")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Configuración: RoBERTa clínico + CrossEntropy ponderada (misma config ganadora)


## 7 · Entrenamiento

In [7]:
def correr_epoca(loader, entrenar=True):
    model.train() if entrenar else model.eval()
    preds,reales,tot=[],[],0.0
    with torch.set_grad_enabled(entrenar):
        for b in loader:
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lab=b['labels'].to(DEVICE)
            if entrenar: optimizer.zero_grad()
            logits=model(ids,mask); loss=criterio(logits,lab)
            if entrenar: loss.backward(); optimizer.step(); scheduler.step()
            tot+=loss.item(); preds.extend(logits.argmax(1).cpu().numpy()); reales.extend(lab.cpu().numpy())
    return tot/len(loader), f1_score(reales,preds,average='macro',zero_division=0)

os.makedirs('../results',exist_ok=True); RUTA='../results/mejor_auditor_sintetico.pt'
print("🏋️  Entrenando (RoBERTa clínico + sintéticos)")
print("Época | Train Loss | Val Loss | Train F1 | Val F1")
mejor,sin=0.0,0
for ep in range(1,EPOCHS+1):
    tl,tf1=correr_epoca(train_loader,True); vl,vf1=correr_epoca(val_loader,False)
    m=""
    if vf1>mejor: mejor,sin=vf1,0; torch.save(model.state_dict(),RUTA); m="  <- mejor"
    else: sin+=1
    print(f"  {ep:2d}  |  {tl:.4f}  |  {vl:.4f}  |  {tf1:.4f}  |  {vf1:.4f}{m}")
    if sin>=PATIENCE: print(f"Early stopping en época {ep}"); break
print(f"🏆 Mejor Val F1 = {mejor:.4f}")

🏋️  Entrenando (RoBERTa clínico + sintéticos)
Época | Train Loss | Val Loss | Train F1 | Val F1
   1  |  1.2290  |  0.8842  |  0.4409  |  0.3379  <- mejor
   2  |  0.6026  |  0.7055  |  0.7347  |  0.4488  <- mejor
   3  |  0.4643  |  0.7396  |  0.7765  |  0.5295  <- mejor
   4  |  0.4043  |  0.6646  |  0.8083  |  0.4632
   5  |  0.3531  |  0.7301  |  0.8274  |  0.5008
   6  |  0.2850  |  0.7224  |  0.8516  |  0.5096
   7  |  0.2119  |  0.8837  |  0.8860  |  0.4181
Early stopping en época 7
🏆 Mejor Val F1 = 0.5295


## 8 · Evaluación en test (7 clases) — test real

In [8]:
model.load_state_dict(torch.load(RUTA,map_location=DEVICE)); model.eval()
p7,r7=[],[]
with torch.no_grad():
    for b in test_loader:
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        p7.extend(model(ids,mask).argmax(1).cpu().numpy()); r7.extend(b['labels'].numpy())
p7,r7=np.array(p7),np.array(r7)
f1m=f1_score(r7,p7,average='macro',zero_division=0); acc=(p7==r7).mean()
print("="*55); print("📊 CON SINTÉTICOS — TEST SET (7 clases, test real)"); print("="*55)
print(f"  Accuracy: {acc:.4f}  |  F1-macro: {f1m:.4f}")
print("="*55)
print(classification_report(r7,p7,target_names=[f'BI-RADS {i}' for i in range(7)],zero_division=0))

📊 CON SINTÉTICOS — TEST SET (7 clases, test real)
  Accuracy: 0.9128  |  F1-macro: 0.5973
              precision    recall  f1-score   support

   BI-RADS 0       0.79      0.90      0.84       145
   BI-RADS 1       0.93      0.98      0.95        89
   BI-RADS 2       0.97      0.94      0.96       396
   BI-RADS 3       0.50      0.15      0.24        13
   BI-RADS 4       1.00      0.25      0.40         8
   BI-RADS 5       0.67      1.00      0.80         2
   BI-RADS 6       0.00      0.00      0.00         1

    accuracy                           0.91       654
   macro avg       0.69      0.60      0.60       654
weighted avg       0.91      0.91      0.91       654



## 9 · Evaluación por zonas (segura 0–3 vs riesgo 4–6)

In [9]:
z=lambda v:(np.asarray(v)>=4).astype(int)
zr,zp=z(r7),z(p7)
tn,fp,fn,tp=confusion_matrix(zr,zp,labels=[0,1]).ravel()
sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
vpp=tp/(tp+fp) if (tp+fp) else 0; f1r=f1_score(zr,zp,pos_label=1,zero_division=0)
print("🛡️  POR ZONAS (test real):")
print(f"  Sensibilidad (recall riesgo): {sens:.4f}")
print(f"  Especificidad:                {espec:.4f}")
print(f"  VPP (precisión riesgo):       {vpp:.4f}")
print(f"  F1 zona de riesgo:            {f1r:.4f}")
print(f"  Riesgo: {tp} detectados / {tp+fn} reales  |  {fn} no detectados  |  {fp} falsas alarmas")

🛡️  POR ZONAS (test real):
  Sensibilidad (recall riesgo): 0.4545
  Especificidad:                1.0000
  VPP (precisión riesgo):       1.0000
  F1 zona de riesgo:            0.6250
  Riesgo: 5 detectados / 11 reales  |  6 no detectados  |  0 falsas alarmas


## 10 · Comparación y decisión

In [10]:
print("F1-macro en test (7 clases, sin fuga, mismo test real):")
print(f"   RoBERTa clínico (sin sintéticos): 0.5871")
print(f"   RoBERTa clínico + sintéticos:     {f1m:.4f}")
delta=f1m-0.5871
print(f"   Diferencia:                       {delta:+.4f}")
print()
if delta>=0.03:
    print("✅ Mejora SUSTANCIAL -> vale la pena conseguir validación clínica (nivel 2).")
elif delta>=0.01:
    print("🟡 Mejora MODERADA -> evaluar si justifica la validación.")
else:
    print("⚪ Mejora marginal/nula -> la generación sintética simple no aporta en este corpus (hallazgo válido).")

F1-macro en test (7 clases, sin fuga, mismo test real):
   RoBERTa clínico (sin sintéticos): 0.5871
   RoBERTa clínico + sintéticos:     0.5973
   Diferencia:                       +0.0102

🟡 Mejora MODERADA -> evaluar si justifica la validación.
